### Degree Program URL Collector

#### Overview

Crawls four Hof University department pages and extracts all degree 
program URLs using department-specific parsing strategies.

- **Input:** Four department page URLs (hardcoded)
- **Tools:** `requests`, `BeautifulSoup`
- **Output:** `hof_university_course_urls.json`

---

#### Departments Crawled

| # | Department | URL |
|---|-----------|-----------------|
| 1 | Business Department | https://www.hof-university.com/about-hof-university/departments/business-department.html
| 2 | Engineering Department | https://www.hof-university.com/about-hof-university/departments/engineering-department.html
| 3 | Computer Science Department | https://www.hof-university.com/about-hof-university/departments/department-for-computer-science.html
| 4 | Graduate School | https://www.hof-university.com/about-hof-university/departments/graduate-school.html

---



In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import re
import time

def fetch_html(url, retries=3, delay=2):
    """
    Fetches HTML content from a given URL with retries and delay,
    including a User-Agent header.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    for i in range(retries):
        try:
            response = requests.get(url, timeout=15, headers=headers)
            response.raise_for_status()
            return response.text
        except requests.exceptions.RequestException as e:
            print(f"Error fetching {url}: {e}")
            if i < retries - 1:
                print(f"Retrying in {delay} seconds...")
                time.sleep(delay)
            else:
                print(f"Failed to fetch {url} after {retries} attempts.")
                return None
    return None

def _extract_business_department_links_internal(soup, base_url):
    """
    Extracts course links specifically from the Business Department page structure.
    """
    course_links = []
    
    bachelor_heading = soup.find('h3', string='Bachelor\'s programs')
    master_heading = soup.find('h3', string='Master\'s programs')

    sections_to_check = []
    if bachelor_heading:
        sections_to_check.append(bachelor_heading.find_next_sibling('div', class_='bodytext'))
    if master_heading:
        sections_to_check.append(master_heading.find_next_sibling('div', class_='bodytext'))
    
    for section in sections_to_check:
        if section:
            for p_tag in section.find_all('p'):
                link = p_tag.find('a', href=True)
                if link:
                    href = link['href']
                    if not href.startswith(('http://', 'https://')):
                        href = base_url.rstrip('/') + href if href.startswith('/') else base_url.rstrip('/') + '/' + href
                    if not href.startswith('mailto:') and not href.endswith(('.pdf', '.doc', '.zip', '.jpg', '.png', '.gif')):
                        course_links.append(href)
    return course_links

def _extract_general_department_links_internal(soup, base_url):
    """
    Extracts course links from department pages like Computer Science and Engineering.
    """
    course_links = []
    
    study_programs_section = None
    for tag in soup.find_all(['h2', 'h3']):
        if 'Our study programs' in tag.get_text(strip=True):
            potential_container = tag.find_next_sibling()
            while potential_container and potential_container.name not in ['div', 'ul', 'ol', 'p']:
                potential_container = potential_container.find_next_sibling()
            
            if potential_container:
                study_programs_section = potential_container
            else:
                parent_div = tag.find_parent('div') 
                if parent_div:
                    study_programs_section = parent_div
            break

    if study_programs_section:
        links = study_programs_section.find_all('a', href=True)
        for link in links:
            href = link['href']
            if href.startswith('/') or base_url in href:
                if not href.startswith(('http://', 'https://')):
                    href = base_url.rstrip('/') + href if href.startswith('/') else base_url.rstrip('/') + '/' + href
                if not href.startswith('mailto:') and not href.endswith(('.pdf', '.doc', '.zip', '.jpg', '.png', '.gif')):
                    course_links.append(href)
    return course_links

def _extract_graduate_school_links_internal(soup, base_url):
    """
    Extracts course links from the Graduate School page.
    Graduate School programs are in <ul class="list-normal"> with direct links.
    """
    course_links = []
    
    # Primary strategy: Find all <ul class="list-normal"> and extract program links
    for ul in soup.find_all('ul', class_='list-normal'):
        for li in ul.find_all('li', recursive=False):
            link = li.find('a', href=True)
            if link:
                href = link['href']
                link_text = link.get_text(strip=True)
                
                # Check if link contains degree indicators (M.B.A., M.Eng., etc.)
                if any(degree in link_text for degree in ['M.B.A', 'M.BA', 'MBA', 'M.Eng', 'M.A.', 'M.Sc', 'Eng.']):
                    # Normalize URL
                    if not href.startswith(('http://', 'https://')):
                        href = base_url.rstrip('/') + href if href.startswith('/') else base_url.rstrip('/') + '/' + href
                    
                    # Exclude non-course links
                    if not href.startswith('mailto:') and not href.endswith(('.pdf', '.doc', '.zip', '.jpg', '.png', '.gif')):
                        course_links.append(href)
    
    # Fallback: Check scholarship-bodytext and bodytext divs
    for section in soup.find_all('div', class_=['scholarship-bodytext', 'bodytext']):
        links = section.find_all('a', href=True)
        for link in links:
            href = link['href']
            link_text = link.get_text(strip=True)
            
            # Only include if it has degree indicators and looks like a program page
            if any(degree in link_text for degree in ['M.B.A', 'M.BA', 'MBA', 'M.Eng', 'M.A.', 'M.Sc']) and \
               ('our-degree-programs' in href or 'studying-at-hof-university' in href):
                if not href.startswith(('http://', 'https://')):
                    href = base_url.rstrip('/') + href if href.startswith('/') else base_url.rstrip('/') + '/' + href
                if not href.startswith('mailto:') and not href.endswith(('.pdf', '.doc', '.zip', '.jpg', '.png', '.gif')):
                    course_links.append(href)
    
    # Remove duplicates while preserving order
    seen = set()
    unique_links = []
    for link in course_links:
        if link not in seen:
            seen.add(link)
            unique_links.append(link)
    
    return unique_links


def extract_course_links(html_content, base_url, dept_url):
    """
    Main function to dispatch to the correct link extraction logic based on department URL.
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    
    if "business-department.html" in dept_url:
        links = _extract_business_department_links_internal(soup, base_url)
    elif "graduate-school.html" in dept_url:
        links = _extract_graduate_school_links_internal(soup, base_url)
    else:
        links = _extract_general_department_links_internal(soup, base_url)
    
    # Print each URL
    print(f"\n  Found {len(links)} links:")
    for idx, link in enumerate(links, 1):
        print(f"    {idx}. {link}")
    
    return links

def main_scraper_urls_only():
    department_urls = [
        "https://www.hof-university.com/about-hof-university/departments/business-department.html",
        "https://www.hof-university.com/about-hof-university/departments/engineering-department.html",
        "https://www.hof-university.com/about-hof-university/departments/department-for-computer-science.html",
        "https://www.hof-university.com/about-hof-university/departments/graduate-school.html"
    ]
    base_hof_url = "https://www.hof-university.com"
    all_unique_course_urls = set()

    print("Starting URL extraction...")
    for dept_url in department_urls:
        
        print(f"\n\n\nProcessing department: {dept_url}")


        dept_html = fetch_html(dept_url)
        if dept_html:
            course_links = extract_course_links(dept_html, base_hof_url, dept_url)
            for link in course_links:
                all_unique_course_urls.add(link)
            time.sleep(1)
        else:
            print(f"Failed to fetch department page: {dept_url}")

    unique_urls_list = list(all_unique_course_urls)

    os.makedirs("data", exist_ok=True)
    os.makedirs("debug_data", exist_ok=True)

    output_filename = "debug_data/hof_university_course_urls.json"
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(unique_urls_list, f, ensure_ascii=False, indent=4)
    
    print(f"URL extraction complete. Total unique course URLs found: {len(unique_urls_list)}")
    print(f"All unique course URLs saved to {output_filename}")
    
if __name__ == "__main__":
    main_scraper_urls_only()

Starting URL extraction...



Processing department: https://www.hof-university.com/about-hof-university/departments/business-department.html

  Found 12 links:
    1. https://www.hof-university.com/studying-at-hof-university/our-degree-programs/business-administration-ba.html
    2. https://www.hof-university.com/studying-at-hof-university/our-degree-programs/business-law-llb.html
    3. https://www.hof-university.com/studying-at-hof-university/our-degree-programs/international-management-ba.html
    4. https://www.hof-university.com/studying-at-hof-university/our-degree-programs/economic-psychology-bsc.html
    5. https://www.hof-university.com/studying-at-hof-university/our-degree-programs/economic-and-organizational-sociology-ba.html
    6. https://www.hof-university.com/studying-at-hof-university/our-degree-programs/supply-chain-management-msc.html
    7. https://www.hof-university.com/studying-at-hof-university/our-degree-programs/marketing-management-msc.html
    8. https://www.

### Degree Program Detail Scraper

#### Overview

Fetches each program URL collected by the URL collector and extracts 
structured details from the program page into a single JSON file.

- **Input:** `hof_university_course_urls.json`
- **Tools:** `requests`, `BeautifulSoup`
- **Output:** `hof_scraped_courses.json`

---

#### Fields Extracted Per Program

| Field | Source on Page |
|-------|---------------|
| `program_name` | Hero section `h1` tag |
| `program_type` | Derived from degree keywords in program name |
| `degree_awarded` | Quick facts box |
| `department` | Quick facts box |
| `duration` | Quick facts box |
| `start` | Quick facts box |
| `language_of_instruction` | Quick facts box |
| `tuition_fees` | Quick facts box |
| `campus` | Quick facts box |
| `application_period` | Quick facts box |
| `admission_requirements` | Accordion section, split into academic and language |
| `course_structure` | Accordion section, parsed by semester |
| `career_perspectives` | Accordion section |
| `contacts` | Staff modal elements |

---



In [6]:
import requests
from bs4 import BeautifulSoup
import json
import re
import time

# ------------------------ Helper Functions ------------------------

def fetch_html(url, retries=3, delay=2):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    for i in range(retries):
        try:
            print(f"Fetching {url} (Attempt {i+1})")
            response = requests.get(url, timeout=15, headers=headers)
            response.raise_for_status()
            return response.text
        except requests.exceptions.RequestException as e:
            print(f"Error fetching {url}: {e}")
            if i < retries - 1:
                time.sleep(delay)
    return None

def extract_employees(html_content: str):
    soup = BeautifulSoup(html_content, "html.parser")
    employees = soup.find_all("div", class_="employee-wrapper")
    contact_dict = {}

    def get_email_from_modal(modal_id):
        modal = soup.find("div", id=modal_id)
        if modal:
            email_tag = modal.find("a", class_="email")
            if email_tag:
                return email_tag.text.replace("(at)", "@").strip()
        return "N/A"

    for emp in employees:
        name_tag = emp.find("a", class_="open-modal")
        name = name_tag.text.strip() if name_tag else "N/A"
        meta_div = emp.find("div", class_="meta")
        designation_tag = meta_div.find("p", class_="mt-3") if meta_div else None
        designation = designation_tag.text.strip() if designation_tag else "N/A"
        phone_tag = emp.find("a", href=lambda x: x and x.startswith("tel:"))
        phone = phone_tag.text.strip() if phone_tag else "N/A"
        modal_id = name_tag.get("data-bs-target", "").replace("#", "") if name_tag else ""
        email = get_email_from_modal(modal_id)
        
        if designation != "N/A":
            # Convert designation to snake_case key
            role_key = designation.lower().replace(' - ', '_').replace(' ', '_').replace('/', '_')
            contact_dict[role_key] = {
                "name": name,
                "phone": phone,
                "email": email
            }
    
    return contact_dict

def extract_course_details(html_content, course_url):
    soup = BeautifulSoup(html_content, 'html.parser')

    course_data = {
        "program_name": None,
        "program_type": None,
        "degree_awarded": None,
        "department": None,
        "duration": None,
        "start": None,
        "language_of_instruction": None,
        "tuition_fees": None,
        "campus": None,
        "program_links": {
            "course_details": course_url,
            "apply": "https://www3.primuss.de/cgi-bin/bew_anmeldung_v3/index.pl?Session=&FH=fhh&Email=&Portal=1&Language=en"
        },
        "admission_requirements": {
            "academic": None,
            "language": None
        },
        "course_structure": {
            "semesters": {}
        },
        "career_perspectives": [],
        "application_period": None,
        "contacts": {}
    }

    # Program Name
    hero_div = soup.find('div', class_='heroteaser-text')
    if hero_div and hero_div.find('h1'):
        course_data['program_name'] = hero_div.find('h1').get_text(strip=True)

    # Quick Facts
    box_yellow = soup.find('div', class_='box-yellow')
    if box_yellow:
        for fact_div in box_yellow.find_all('div', recursive=False):
            span_tag = fact_div.find('span', recursive=False)
            if not span_tag: continue
            key = span_tag.get_text(strip=True).lower()
            temp_div = BeautifulSoup(str(fact_div), 'html.parser')
            temp_div.span.decompose()
            value = temp_div.get_text(separator=' ', strip=True)

            if "degree" in key: course_data["degree_awarded"] = value
            elif "department" in key: course_data["department"] = value
            elif "duration" in key: course_data["duration"] = value
            elif "start" in key: course_data["start"] = value
            elif "language" in key: course_data["language_of_instruction"] = value
            elif "tuition" in key: course_data["tuition_fees"] = value
            elif "campus" in key: course_data["campus"] = value
            elif "application period" in key: course_data["application_period"] = value

   # Program Type
    if course_data.get("program_name"):
        masters_keywords = ["M.Sc", "Master", "M.Eng./M.A.", "M.Eng.", "M.A.", "(M.Sc.)", "M.B.A. and Eng.","M.B.A.", "M. Eng."]
        bachelors_keywords = ["B.Sc", "Bachelor", "LL.B", "(B.Eng.)", "B.Eng.", "B.A."]

        if any(keyword in course_data["program_name"] for keyword in masters_keywords):
            course_data["program_type"] = "Master"
        elif any(keyword in course_data["program_name"] for keyword in bachelors_keywords):
            course_data["program_type"] = "Bachelor"


    # Accordion sections
    accordion_box = soup.find('div', class_='box-accordion')
    if accordion_box:
        for item in accordion_box.find_all('div', class_='accordion-item'):
            header = item.find('h4', class_='accordion-header')
            content = item.find('div', class_='accordion-collapse')
            if not header or not content: 
                continue
            
            section_title = header.get_text(strip=True).lower()
            text = content.get_text(separator=' ', strip=True)

            if "admission requirements" in section_title:
                # Split by "Language requirements" (case-insensitive)
                if "language requirements" in text.lower():
                    # Find the position of "Language requirements"
                    lang_pos = text.lower().find("language requirements")
                    
                    # Everything before is academic requirements
                    course_data["admission_requirements"]["academic"] = text[:lang_pos].strip()
                    
                    # Everything from "Language requirements" onward is language
                    course_data["admission_requirements"]["language"] = text[lang_pos:].strip()
                else:
                    # No language section found, everything is academic
                    course_data["admission_requirements"]["academic"] = text
                    
            elif "course structure" in section_title:
                lines = text.split("\n")
                semester_data = {}
                current_sem = None
                for line in lines:
                    if re.search(r"semester\s*\d+", line, re.IGNORECASE):
                        current_sem = re.findall(r"semester\s*\d+", line, re.IGNORECASE)[0]
                        semester_data[current_sem] = []
                    elif current_sem:
                        semester_data[current_sem].append(line.strip())
                for k, v in semester_data.items():
                    course_data["course_structure"]["semesters"][k] = v
                    
            elif "career perspectives" in section_title or "job profile" in section_title:
                course_data["career_perspectives"] = [s.strip() for s in text.split("•") if s.strip()]

    # Contacts
    course_data["contacts"] = extract_employees(html_content)

    return course_data

# ------------------------ Main Scraping ------------------------

def main_scraper():
    with open("debug_data/hof_university_course_urls.json", "r", encoding="utf-8") as f:
        course_urls = json.load(f)

    all_course_details = {}

    for url in course_urls:
        html_content = fetch_html(url)
        if html_content:
            details = extract_course_details(html_content, url)
            if details and details.get("program_name"):
                all_course_details[details["program_name"]] = details
            time.sleep(1)  # polite delay

 

    with open("debug_data/hof_scraped_courses.json", "w", encoding="utf-8") as f:
        json.dump(all_course_details, f, ensure_ascii=False, indent=4)

    print("Scraped course data saved to debug_data/hof_scraped_courses.json")


if __name__ == "__main__":
    main_scraper()


Fetching https://www.hof-university.com/studying-at-hof-university/our-degree-programs/supply-chain-management-msc.html (Attempt 1)
Fetching https://www.hof-university.com/studying-at-hof-university/our-degree-programs/business-administration-ba.html (Attempt 1)
Fetching https://www.hof-university.com/studying-at-hof-university/our-degree-programs/ai-driven-supply-chain-management-msc.html (Attempt 1)
Fetching https://www.hof-university.com/studying-at-hof-university/our-degree-programs/computer-science-international-bsc.html (Attempt 1)
Fetching https://www.hof-university.com/studying-at-hof-university/software-engineering-for-industrial-applications-meng.html (Attempt 1)
Fetching https://www.hof-university.com/studying-at-hof-university/our-degree-programs/digitalization-and-innovation-mba.html (Attempt 1)
Fetching https://www.hof-university.com/studying-at-hof-university/our-degree-programs/general-management-mba.html (Attempt 1)
Fetching https://www.hof-university.com/studying-at-h

### Eligible Backgrounds Mapping

#### Overview

Defines a manual mapping of each master's program to its eligible 
applicant backgrounds and merges this into the scraped program data.

- **Input:** Hardcoded mapping + `hof_scraped_courses.json`
- **Tools:** `json`
- **Output:** `hof_merged_courses.json`






---
> **Note:** This hardcoded mapping was only used in the without-memory 
> pipeline. In the memory pipeline, an LLM-based query router was 
> introduced to handle eligibility queries dynamically, replacing the 
> need for manual background mappings.
---





In [7]:
import json

# Simple mapping: program_name → eligible_backgrounds
PROGRAM_BACKGROUNDS = {
    # MASTER - BUSINESS
    "Global Management M.A.": {
        "eligible_backgrounds": ["Business Management", "International Management", "Industrial Engineering", "Business Informatics"]
    },
    "AI-Driven Supply Chain Management (M.Sc.)": {
        "eligible_backgrounds": ["Business Management", "International Management", "Industrial Engineering", "Business Informatics"]
    },
    "Digital Business Management (M.Sc.)": {
        "eligible_backgrounds": ["Business Administration", "Economics", "Information Technology"]
    },
    "Applied Psychology M.Sc.": {
        "eligible_backgrounds": ["Psychology", "Business Psychology"]
    },
    "Supply Chain Management M.Sc.": {
        "eligible_backgrounds": ["Business Management", "Industrial Engineering", "Engineering", "Business Information Systems"]
    },
    
    # MASTER - COMPUTER SCIENCE
    "Computer Science M.Sc.": {
        "eligible_backgrounds": ["Computer Science"]
    },
    "Applied Research in Computer Science M.Sc.": {
        "eligible_backgrounds": ["Computer Science", "Media Informatics", "Mobile Computing", "Business Information Systems"]
    },
    "Artificial Intelligence and Robotics M.Sc.": {
        "eligible_backgrounds": ["Computer Science", "Engineering", "Natural Sciences"]
    },
    
    # MASTER - ENGINEERING
    "Textile Design M.A.": {
        "eligible_backgrounds": ["Design", "Architecture"]
    },
    "Sustainable Textiles M.Eng.": {
        "eligible_backgrounds": ["Engineering", "Natural Sciences"]
    },
    "Sustainable Water Management and Engineering M.Eng.": {
        "eligible_backgrounds": ["Engineering", "Natural Sciences"]
    },
    "Artificial Intelligence Aided Mobility Design M.A.": {
        "eligible_backgrounds": ["Engineering", "Design"]
    },
    
    # MASTER - GRADUATE SCHOOL
    "Operational Excellence M.B.A. and Eng.": {
        "eligible_backgrounds": ["Engineering Sciences"]
    },
    "Software Engineering for Industrial Applications M.Eng.": {
        "eligible_backgrounds": ["Software Engineering"]
    },
    "Sustainable Engineering and Project Management  M.B.A. and Eng.": {
        "eligible_backgrounds": ["Engineering Sciences"]
    },
    "General Management M.B.A.": {
        "eligible_backgrounds": ["Humanities", "Social Sciences", "Engineering Sciences", "Economic Sciences"]
    },
    "Digitalization and Innovation M.B.A.": {
        "eligible_backgrounds": ["Engineering", "Humanities", "Social Sciences", "Business Administration", "Economics", "Computer Science"]
    },
    

}

# Save to file
if __name__ == "__main__":
    with open('debug_data/program_background_mapping.json', 'w', encoding='utf-8') as f:
        json.dump(PROGRAM_BACKGROUNDS, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Saved {len(PROGRAM_BACKGROUNDS)} programs")

✅ Saved 17 programs


### Merge program_background_mapping.json with hof_scraped_courses.json data

In [8]:
import json

def merge_json_files(hof_file, eligibility_file, output_file):
    # Load Hof scraped courses
    with open(hof_file, "r", encoding="utf-8") as f:
        hof_data = json.load(f)

    # Load eligibility data
    with open(eligibility_file, "r", encoding="utf-8") as f:
        eligibility_data = json.load(f)

    # Merge eligibility info into hof_data
    for program, eligibility in eligibility_data.items():
        if program in hof_data:
            hof_data[program].update(eligibility)
        else:
            # If program not in hof_data, add it entirely
            hof_data[program] = eligibility

    # Save merged output
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(hof_data, f, indent=4, ensure_ascii=False)

    print(f"Merged data saved to {output_file}")


# Example usage
merge_json_files("debug_data/hof_scraped_courses.json", "debug_data/program_background_mapping.json", "debug_data/hof_merged_courses.json")


Merged data saved to debug_data/hof_merged_courses.json


### SPO Merger

#### Overview

Matches each degree program to its corresponding SPO document using 
normalized name matching and merges module structures and 
program-specific regulation sections into the program records.

- **Input:** `hof_merged_courses.json`, `spo_clean_data.json`
- **Tools:** `json`, `re`
- **Output:** `all_programs_data.json`


In [10]:
import json
from typing import Dict, Any
import re


def normalize_section_name(section_title: str) -> str:
    """
    Normalize section titles to standard keys for the merged JSON.
    """
    title_lower = section_title.lower().strip()
    
    # Master thesis variations
    master_thesis_keywords = ['master thesis', "master's thesis", 'masters thesis']
    if any(keyword in title_lower for keyword in master_thesis_keywords):
        if 'internship' in title_lower:
            return 'internship_and_master_thesis'
        return 'master_thesis'
    
    # Bachelor thesis variations
    bachelor_thesis_keywords = ['bachelor thesis', "bachelor's thesis", 'bachelors thesis']
    if any(keyword in title_lower for keyword in bachelor_thesis_keywords):
        return 'bachelor_thesis'
    
    # Final theses (could be either bachelor or master)
    if 'final theses' in title_lower or 'final thesis' in title_lower:
        return 'final_thesis'
    
    # Internship variations
    if title_lower in ['internship', 'internships']:
        return 'internship'
    
    if 'internship report' in title_lower:
        return 'internship_report'
    
    # Practical semester
    if 'practical semester' in title_lower:
        return 'practical_semester'
    
    # Compulsory elective modules
    if 'compulsory elective' in title_lower:
        return 'compulsory_elective_modules'
    
    # Third re-examinations
    if 'third re-examination' in title_lower:
        return 'third_re_examinations'
    
    # All admission requirements variations
    admission_keywords = ['admission requirement', 'admission requirements', 
                         'specific admission requirement']
    if any(keyword in title_lower for keyword in admission_keywords):
        return 'detailed_admission_requirements'
    
    # If no match found, create a safe key name
    safe_name = section_title.lower().strip()
    safe_name = safe_name.replace(' ', '_')
    safe_name = safe_name.replace("'", '')
    safe_name = safe_name.replace('-', '_')
    return safe_name


def normalize_for_matching(name: str) -> str:
    """Normalize program name for matching"""
    name = name.lower().strip()
    # Remove parentheses but keep contents
    name = name.replace('(', ' ').replace(')', ' ')
    # Remove dots
    name = name.replace('.', '')
    # Handle variations
    name = name.replace('textilien', 'textiles')
    # Normalize spaces
    name = ' '.join(name.split())
    
    return name


def find_best_spo_match(program_name: str, spo_data: Dict[str, Any]) -> tuple:
    """Find exact or very close SPO match"""
    program_norm = normalize_for_matching(program_name)
    
    # First try: exact match after normalization
    for spo_name, spo_info in spo_data.items():
        spo_norm = normalize_for_matching(spo_name)
        
        if program_norm == spo_norm:
            return (spo_name, spo_info)
    
    # Second try: extract key parts and match
    program_words = program_norm.split()
    
    # Extract degree level
    program_degree = None
    for word in program_words:
        if word in ['bsc', 'beng', 'ba', 'msc', 'meng', 'ma', 'mba', 'llm']:
            program_degree = word
            break
        if 'mba and eng' in program_norm:
            program_degree = 'mba and eng'
            break
    
    # Get program name without degree
    degree_words = ['bsc', 'beng', 'ba', 'msc', 'meng', 'ma', 'mba', 'llm', 'and', 'eng']
    program_core_words = [w for w in program_words if w not in degree_words]
    
    best_match = None
    best_score = 0
    
    for spo_name, spo_info in spo_data.items():
        spo_norm = normalize_for_matching(spo_name)
        spo_words = spo_norm.split()
        
        # Extract SPO degree
        spo_degree = None
        for word in spo_words:
            if word in ['bsc', 'beng', 'ba', 'msc', 'meng', 'ma', 'mba', 'llm']:
                spo_degree = word
                break
        if 'mba and eng' in spo_norm:
            spo_degree = 'mba and eng'
        
        # CRITICAL: Degrees must match (B.Sc. should not match M.Sc.)
        if program_degree and spo_degree and program_degree != spo_degree:
            # Allow some flexibility: B.Eng can match B.Sc.
            if not (program_degree in ['bsc', 'beng'] and spo_degree in ['bsc', 'beng']):
                continue
        
        # Get SPO core words
        spo_core_words = [w for w in spo_words if w not in degree_words]
        
        # Count matching core words
        common_core = set(program_core_words) & set(spo_core_words)
        
        # Must have at least 2 common core words AND most words must match
        if len(common_core) >= 2:
            # Calculate match ratio
            total_core = len(set(program_core_words) | set(spo_core_words))
            match_ratio = len(common_core) / total_core if total_core > 0 else 0
            
            # Require at least 70% match on core words
            if match_ratio >= 0.7:
                score = len(common_core) + (10 if program_degree == spo_degree else 0)
                
                if score > best_score:
                    best_score = score
                    best_match = (spo_name, spo_info)
    
    return best_match if best_match else (None, None)


def main():
    """Merge SPO data into existing program data"""
    
    print("Loading existing program data...")
    with open('debug_data/hof_merged_courses.json', 'r', encoding='utf-8') as f:
        program_data = json.load(f)
    
    print("Loading SPO data...")
    with open('debug_data/spo_clean_data.json', 'r', encoding='utf-8') as f:
        spo_data = json.load(f)
    
    print(f"\nPrograms: {len(program_data)}")
    print(f"SPO documents: {len(spo_data)}\n")
    
    print("MERGING SPO DATA INTO PROGRAMS\n")
    
    matched_count = 0
    matched_programs = []
    unmatched_programs = []
    
    for program_name, program_info in program_data.items():
        spo_name, spo_info = find_best_spo_match(program_name, spo_data)
        
        if spo_info:
            matched_count += 1
            matched_programs.append(program_name)
            
            # Add modules
            if 'modules' in spo_info and spo_info['modules']:
                program_info['modules'] = spo_info['modules']
                print(f"{program_name}")
        
            
            # Add sections
            if 'sections' in spo_info and spo_info['sections']:
                sections_added = []
                for section_title, section_content in spo_info['sections'].items():
                    normalized_key = normalize_section_name(section_title)
                    program_info[normalized_key] = section_content
                    sections_added.append(normalized_key)
                
                print(f"Added modules:  {', '.join(sections_added)}")
            
            print()
        else:
            unmatched_programs.append(program_name)
    
    # Save
    output_file = 'data/all_programs_data.json'
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(program_data, f, indent=2, ensure_ascii=False)
    
    print("="*80)
    print("SUMMARY")
    print("="*80)
    print(f"Matched: {matched_count}/{len(spo_data)} SPO documents")
    print(f"Programs with modules: {sum(1 for p in program_data.values() if 'modules' in p)}")
    
    # Show unused SPO
    used_spo = set()
    for prog_name in matched_programs:
        spo_name, _ = find_best_spo_match(prog_name, spo_data)
        if spo_name:
            used_spo.add(spo_name)
    
    unused_spo = set(spo_data.keys()) - used_spo
    if unused_spo:
        print(f"\n Unused SPO documents: {len(unused_spo)}")
        for spo in unused_spo:
            print(f"  • {spo}")
    
    print(f"\nSaved to: {output_file}\n")


if __name__ == "__main__":
    main()

Loading existing program data...
Loading SPO data...

Programs: 37
SPO documents: 14

MERGING SPO DATA INTO PROGRAMS

Software Engineering for Industrial Applications M.Eng.
Added modules:  detailed_admission_requirements, master_thesis

Digitalization and Innovation M.B.A.
Added modules:  detailed_admission_requirements, internship, master_thesis

General Management M.B.A.
Added modules:  detailed_admission_requirements, internship, master_thesis

Sustainable Textiles M.Eng.
Added modules:  detailed_admission_requirements, master_thesis, language_of_instruction_and_examination

Sustainable Engineering and Project Management  M.B.A. and Eng.
Added modules:  detailed_admission_requirements, compulsory_elective_modules, internship, master_thesis

Artificial Intelligence and Robotics M.Sc.
Added modules:  detailed_admission_requirements, master_thesis

Sustainable Water Management and Engineering M.Eng.
Added modules:  detailed_admission_requirements, master_thesis

Computer Science (B.Sc